# **COMBINE CSV FILES**

In [ ]:
#COMBINE CSV FILES

import pandas as pd

file_names = [
"official-out1.csv", "official-out2.csv", "official-out3.csv",
"official-out4.csv", "official-out5.csv", "official-out6.csv",
"official-out7.csv", "official-out8.csv", "official-out9.csv",
"official-out10.csv", "official-out11.csv", "official-out12.csv",
"official-out13.csv", "official-out14.csv"]

combined_df = pd.DataFrame()


for i, file_name in enumerate(file_names):
    file_path = f'{file_name}'
    temp_df = pd.read_csv(file_path)
    combined_df = pd.concat([combined_df, temp_df], ignore_index=True)
combined_df.to_csv('combined_file.csv', index=False)

**COMBINE TXT FILES**

In [ ]:
def merge_files(input_files, output_file):
    with open(output_file, 'w') as outfile:
        for input_file in input_files:
            with open(input_file, 'r') as infile:
                outfile.write(infile.read())

# List of input files
input_files = [
    "merging_result_off.txt",
    "merging_result_off1_first_10_.txt",
    "merging_result_off1_last_12_.txt",
    "merging_result_off2.txt"]

# Output file
output_file = "combined_merging.txt"

# Merge files
merge_files(input_files, output_file)

print("Files merged successfully!")



Files merged successfully!


**MERGE EDGES WITH CSV FILE**

In [ ]:
#MERGE EDGES WITH CSV FILE

import csv

input_file_path = 'official-1-out1.csv'
edge_output_path = 'edge_output.txt'
updated_edge_output_path = 'updated_edge_output.txt'
merging_result_path = 'merging_result_TOTAL.txt'

# Step 1: Read the CSV and format edges
edges = []
with open(input_file_path, mode='r', newline='') as file:
    reader = csv.DictReader(file)
    for row in reader:
        formatted_line = f"Src_Node_Subject_Name: {row['subjectname']} Src_Node_Subject_Type: {row['subject_type']} Dest_Node_Object_Name: {row['objectname']} Dest_Node_Object_Type: {row['object_type']} Syscall: {row['syscall']} Timestamp: {row['timestamp']}"
        edges.append(formatted_line)

# Write the formatted edges to a file
with open(edge_output_path, 'w') as file:
    for line in edges:
        file.write(f"{line}\n")

# Step 2: Calculate minimum timestamps for each destination node
dest_node_timestamps = {}
for line in edges:
    parts = line.split(' ')
    dest_node_name = ' '.join(parts[parts.index('Dest_Node_Object_Name:') + 1:parts.index('Syscall:')])
    timestamp = int(parts[-1])
    if dest_node_name not in dest_node_timestamps or timestamp < dest_node_timestamps[dest_node_name]:
        dest_node_timestamps[dest_node_name] = timestamp

# Step 3: Update edges with time difference
updated_edges = []
for line in edges:
    parts = line.split(' ')
    dest_node_name = ' '.join(parts[parts.index('Dest_Node_Object_Name:') + 1:parts.index('Syscall:')])
    timestamp_index = parts.index('Timestamp:') + 1
    timestamp = int(parts[timestamp_index])
    time_difference = timestamp - dest_node_timestamps[dest_node_name]
    # Exclude 'Timestamp' from the updated line
    updated_line = ' '.join(parts[:timestamp_index - 1]) + f" Time_Difference: [{time_difference}, {time_difference}]"
    updated_edges.append(updated_line)


# Write the updated edges to a file
with open(updated_edge_output_path, 'w') as file:
    for line in updated_edges:
        file.write(f"{line}\n")

# Step 4: Merge edges and provide detailed output
initial_edge_count = len(updated_edges)
merging_operations = 0
merged_edges = {}

for line in updated_edges:
    # Identify the edge without the time difference
    edge_key = ' '.join(line.split(' ')[:-3])
    time_diffs = eval(line.split('Time_Difference: ')[-1])

    if edge_key in merged_edges:
        #print("Merging following edges:")
        #print(f"{edge_key} Time_Difference: {merged_edges[edge_key]}")
        #print(f"{edge_key} Time_Difference: {time_diffs}")
        merging_operations += 1

        # Merge time differences
        merged_time_diff = [min(merged_edges[edge_key][0], time_diffs[0]), max(merged_edges[edge_key][1], time_diffs[1])]
        #print()
        #print("Resulting merged edge:")
        #print(f"{edge_key} Time_Difference: {merged_time_diff}\n")
        merged_edges[edge_key] = merged_time_diff
    else:
        merged_edges[edge_key] = time_diffs

final_edge_count = len(merged_edges)

# Write the merged edges to a file
with open(merging_result_path, 'w') as file:
    for edge, time_diff in merged_edges.items():
        file.write(f"{edge} Time_Difference: {time_diff}\n")

# Print the edge and merging counts
print(f"Total number of initial edges: {initial_edge_count}")
print(f"Total number of merging operations: {merging_operations}")
print(f"Total number of final edges: {final_edge_count}")

Total number of initial edges: 227088
Total number of merging operations: 221549
Total number of final edges: 5539


**MERGE EDGES WITH TXT FILE**

In [ ]:
def merge_edges_and_print_details(edge_lines):
    merged_edges = {}
    merging_operations = 0

    for line in edge_lines:
        parts = line.strip().split(' Time_Difference: ')
        edge_key = parts[0]
        time_diff = eval(parts[1])

        if edge_key in merged_edges:
            # merging edges
            combined_time_diff = [
                min(merged_edges[edge_key][0], time_diff[0]),
                max(merged_edges[edge_key][1], time_diff[1]),
            ]

            merged_edges[edge_key] = combined_time_diff
            merging_operations += 1
        else:
            merged_edges[edge_key] = time_diff

    return merged_edges, merging_operations


with open('combined_merging.txt', 'r') as file:
    lines = file.readlines()

initial_edge_count = len(lines)
total_merging_operations = 0

#merge the edges
while True:
    new_edges, merging_operations = merge_edges_and_print_details(lines)
    total_merging_operations += merging_operations
    new_lines = [f"{edge} Time_Difference: {time_diff}" for edge, time_diff in new_edges.items()]
    if new_lines == lines:  # No more merges possible
        break
    lines = new_lines

final_edge_count = len(lines)

#write the result to a txt file
with open('merging_result_TOTAL.txt', 'w') as file:
    for line in lines:
        file.write(f"{line}\n")

print(f"Total number of initial edges: {initial_edge_count}")
print(f"Total number of merging operations: {total_merging_operations}")
print(f"Total number of final edges: {final_edge_count}")


Total number of initial edges: 1005418
Total number of merging operations: 530507
Total number of final edges: 474911
